In [1]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import KernelPCA
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# --- 1. CLASS TARGET ENCODER TÙY CHỈNH ---
# Giúp xử lý các biến chữ có quá nhiều giá trị (như Tên cây trồng, Tên quận)
# bằng cách thay thế chúng bằng giá trị trung bình của Yield tương ứng.
class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.maps = {}
        self.global_mean = 0

    def fit(self, X, y):
        # Lưu lại giá trị trung bình toàn cục để điền khuyết
        self.global_mean = y.mean()
        # Duyệt qua từng cột được đưa vào transformer này
        for col in X.columns:
            # Tính trung bình Yield cho từng nhóm
            mapper = y.groupby(X[col]).mean()
            self.maps[col] = mapper
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in X.columns:
            # Map giá trị trung bình vào cột
            X_copy[col] = X_copy[col].map(self.maps.get(col, {}))
            # Điền giá trị thiếu bằng trung bình toàn cục
            X_copy[col] = X_copy[col].fillna(self.global_mean)
        return X_copy

# --- 2. ĐỌC DỮ LIỆU ---
df = pd.read_csv('Agri_Data_Cleaned.csv')

# --- 3. TỰ ĐỘNG PHÂN LOẠI BIẾN ---

# Loại bỏ các cột gây rò rỉ dữ liệu (Data Leakage) hoặc không cần thiết
target_col = 'Yield'
drop_cols = ['Yield', 'Production', 'AP Ratio'] 
cols_to_drop = [c for c in drop_cols if c in df.columns]

X = df.drop(columns=cols_to_drop)
y = df[target_col]

# Tách biến số và biến chữ
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Tự động chia nhóm biến chữ dựa trên số lượng giá trị duy nhất (Cardinality)
# Ngưỡng (Threshold) = 20: Nếu cột có > 20 giá trị khác nhau -> Coi là phức tạp (High Cardinality)
threshold = 20
high_card_features = [col for col in categorical_features if X[col].nunique() > threshold]
low_card_features = [col for col in categorical_features if X[col].nunique() <= threshold]

print(f"🔹 Biến số ({len(numeric_features)}): {numeric_features[:5]}...")
print(f"🔹 Biến chữ phức tạp (> {threshold} giá trị) -> Dùng Target Encoding: {high_card_features}")
print(f"🔹 Biến chữ đơn giản (<= {threshold} giá trị) -> Dùng One-Hot Encoding: {low_card_features}")

# --- 4. XÂY DỰNG PIPELINE XỬ LÝ TỰ ĐỘNG ---

# A. Pipeline cho biến số: Điền khuyết + Chuẩn hóa
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# B. Pipeline cho biến chữ phức tạp (High Cardinality): Target Encoding + Chuẩn hóa
high_card_transformer = Pipeline(steps=[
    ('target_enc', TargetEncoder()), 
    ('scaler', StandardScaler())
])

# C. Pipeline cho biến chữ đơn giản (Low Cardinality): Điền khuyết + One-Hot Encoding
low_card_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Tổng hợp các bước xử lý cột
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('high_card', high_card_transformer, high_card_features),
        ('low_card', low_card_transformer, low_card_features)
    ])

# --- 5. PIPELINE CUỐI CÙNG VỚI KERNEL PCA ---
# Kernel PCA giúp "gỡ rối" các mối quan hệ phi tuyến tính phức tạp
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('kpca', KernelPCA(n_components=15, kernel='rbf', gamma=None)) 
])

# --- 6. THỰC THI VÀ KIỂM TRA ---
# Chia tập train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit pipeline trên tập train
# Lưu ý: Target Encoding cần cả X và y để học mối quan hệ
X_train_processed = full_pipeline.fit_transform(X_train, y_train)
X_test_processed = full_pipeline.transform(X_test)

# Hiển thị kết quả
print("\n✅ Kích thước dữ liệu gốc:", X_train.shape)
print("✅ Kích thước dữ liệu sau khi xử lý (đã giảm chiều & khử phi tuyến):", X_train_processed.shape)

# Xem 5 dòng đầu của dữ liệu mới (Các thành phần chính - Principal Components)
df_new = pd.DataFrame(X_train_processed, columns=[f'PC{i+1}' for i in range(15)])
print("\nDữ liệu mới sẵn sàng cho mô hình:")
print(df_new.head())

🔹 Biến số (39): ['Area', 'Avg Temp', 'Avg Humidity', 'Max Temp', 'Min Temp']...
🔹 Biến chữ phức tạp (> 20 giá trị) -> Dùng Target Encoding: ['District', 'Crop Name', 'Growth', 'Harvest']
🔹 Biến chữ đơn giản (<= 20 giá trị) -> Dùng One-Hot Encoding: ['Season', 'Transplant', 'pH_Suitability', 'Dominant_Soil_Texture', 'Water_Availability_Cat', 'Extreme_Heat_Risk']

✅ Kích thước dữ liệu gốc: (3342, 49)
✅ Kích thước dữ liệu sau khi xử lý (đã giảm chiều & khử phi tuyến): (3342, 15)

Dữ liệu mới sẵn sàng cho mô hình:
        PC1       PC2       PC3       PC4       PC5       PC6       PC7  \
0 -0.131752  0.401821  0.070210  0.082824  0.011443 -0.077822  0.108565   
1  0.335350  0.003660 -0.110704  0.290224 -0.025137 -0.154966 -0.234778   
2  0.152546  0.253126 -0.178880 -0.076932 -0.188995 -0.145388  0.030692   
3  0.216843 -0.020875  0.035171  0.124196 -0.029430  0.032209 -0.109248   
4 -0.301014 -0.333276  0.215006 -0.041942  0.173841 -0.078754  0.017547   

        PC8       PC9      PC10  